# Practical 3: Implement Adversarial Search with Alpha-Beta Pruning for Game Playing

## Aim
Implement Alpha-Beta pruning to reduce unnecessary exploration during adversarial game-tree search.

In [1]:
# Cell 1: Game Tree Representation, Minimax, and Alpha-Beta Pruning Implementation

class GameTreeNode:
    """Represents a node in a deterministic adversarial game tree."""
    def __init__(self, name, value=None, children=None):
        self.name = name
        self.value = value  # Numeric payoff if leaf node, else None
        self.children = children if children is not None else []

    def is_leaf(self):
        return len(self.children) == 0

def minimax(node, depth, is_maximizing, stats):
    """Standard Minimax traversal evaluating all reachable leaf nodes."""
    if node.is_leaf():
        stats['leaf_evaluations'] += 1
        return node.value

    if is_maximizing:
        max_eval = float('-inf')
        for child in node.children:
            eval_val = minimax(child, depth + 1, False, stats)
            max_eval = max(max_eval, eval_val)
        return max_eval
    else:
        min_eval = float('inf')
        for child in node.children:
            eval_val = minimax(child, depth + 1, True, stats)
            min_eval = min(min_eval, eval_val)
        return min_eval

def alphabeta(node, depth, alpha, beta, is_maximizing, stats, pruned_records):
    """Adversarial search with Alpha-Beta bounds and subtree pruning."""
    if node.is_leaf():
        stats['leaf_evaluations'] += 1
        return node.value

    if is_maximizing:
        max_eval = float('-inf')
        for idx, child in enumerate(node.children):
            eval_val = alphabeta(child, depth + 1, alpha, beta, False, stats, pruned_records)
            max_eval = max(max_eval, eval_val)
            alpha = max(alpha, eval_val)
            if beta <= alpha:
                # Pruning condition triggered: beta <= alpha
                unvisited = [c.name for c in node.children[idx + 1:]]
                if unvisited:
                    pruned_records.append({
                        'at_node': node.name,
                        'player': 'MAX',
                        'pruned_subtrees': unvisited,
                        'condition': f"beta ({beta}) <= alpha ({alpha})"
                    })
                break
        return max_eval
    else:
        min_eval = float('inf')
        for idx, child in enumerate(node.children):
            eval_val = alphabeta(child, depth + 1, alpha, beta, True, stats, pruned_records)
            min_eval = min(min_eval, eval_val)
            beta = min(beta, eval_val)
            if beta <= alpha:
                # Pruning condition triggered: beta <= alpha
                unvisited = [c.name for c in node.children[idx + 1:]]
                if unvisited:
                    pruned_records.append({
                        'at_node': node.name,
                        'player': 'MIN',
                        'pruned_subtrees': unvisited,
                        'condition': f"beta ({beta}) <= alpha ({alpha})"
                    })
                break
        return min_eval


In [2]:
# Cell 2: Execution, Pruning Trace, and Algorithm Comparison

def build_deterministic_tree():
    """
    Constructs a deterministic depth-3 game tree:
    Root (MAX)
      ├── Node A (MIN)
      │     ├── Node C (MAX) -> Leaves L1 (3), L2 (5)
      │     └── Node D (MAX) -> Leaves L3 (6), L4 (9)
      └── Node B (MIN)
            ├── Node E (MAX) -> Leaves L5 (1), L6 (2)
            └── Node F (MAX) -> Leaves L7 (0), L8 (-1) [Pruned by Alpha-Beta]
    """
    L1 = GameTreeNode("L1", value=3)
    L2 = GameTreeNode("L2", value=5)
    L3 = GameTreeNode("L3", value=6)
    L4 = GameTreeNode("L4", value=9)

    L5 = GameTreeNode("L5", value=1)
    L6 = GameTreeNode("L6", value=2)
    L7 = GameTreeNode("L7", value=0)
    L8 = GameTreeNode("L8", value=-1)

    C = GameTreeNode("C", children=[L1, L2])
    D = GameTreeNode("D", children=[L3, L4])
    E = GameTreeNode("E", children=[L5, L6])
    F = GameTreeNode("F", children=[L7, L8])

    A = GameTreeNode("A", children=[C, D])
    B = GameTreeNode("B", children=[E, F])

    root = GameTreeNode("Root", children=[A, B])
    return root

# 1. Execute standard Minimax
tree_for_minimax = build_deterministic_tree()
minimax_stats = {'leaf_evaluations': 0}
minimax_val = minimax(tree_for_minimax, 0, True, minimax_stats)

# 2. Execute Alpha-Beta Pruning
tree_for_alphabeta = build_deterministic_tree()
alphabeta_stats = {'leaf_evaluations': 0}
pruned_records = []
alphabeta_val = alphabeta(
    tree_for_alphabeta, 0, float('-inf'), float('inf'), True, alphabeta_stats, pruned_records
)

print("=" * 65)
print("ADVERSARIAL SEARCH: MINIMAX VS ALPHA-BETA PRUNING")
print("=" * 65)
print("1. Standard Minimax Evaluation:")
print(f"   - Root Minimax Value     : {minimax_val}")
print(f"   - Total Leaf Evaluations : {minimax_stats['leaf_evaluations']}")
print("-" * 65)
print("2. Alpha-Beta Pruning Evaluation:")
print(f"   - Root Alpha-Beta Value  : {alphabeta_val}")
print(f"   - Total Leaf Evaluations : {alphabeta_stats['leaf_evaluations']}")
print(f"   - Branches Pruned Count  : {len(pruned_records)}")
for idx, pr in enumerate(pruned_records, 1):
    print(f"     Pruning Event {idx}: At {pr['at_node']} ({pr['player']}) | Pruned: {pr['pruned_subtrees']} | Trigger: {pr['condition']}")

print("-" * 65)
print("Comparative Performance Summary:")
print(f"{'Algorithm':<20} | {'Root Score':<12} | {'Leaves Evaluated':<18} | {'Leaves Pruned':<12}")
print("-" * 65)
print(f"{'Minimax':<20} | {minimax_val:<12} | {minimax_stats['leaf_evaluations']:<18} | 0")
leaves_saved = minimax_stats['leaf_evaluations'] - alphabeta_stats['leaf_evaluations']
print(f"{'Alpha-Beta Pruning':<20} | {alphabeta_val:<12} | {alphabeta_stats['leaf_evaluations']:<18} | {leaves_saved}")
print("-" * 65)

# Verification checks
assert minimax_val == alphabeta_val, "Verification Failed: Root values differ!"
assert alphabeta_stats['leaf_evaluations'] < minimax_stats['leaf_evaluations'], "Verification Failed: No pruning occurred!"
print("Verification Checks:")
print(f" [PASS] Root minimax scores match: {minimax_val == alphabeta_val}")
print(f" [PASS] Alpha-Beta evaluated fewer leaves: {alphabeta_stats['leaf_evaluations']} < {minimax_stats['leaf_evaluations']}")
print("Verification Status: ALL CHECKS PASSED")


ADVERSARIAL SEARCH: MINIMAX VS ALPHA-BETA PRUNING
1. Standard Minimax Evaluation:
   - Root Minimax Value     : 5
   - Total Leaf Evaluations : 8
-----------------------------------------------------------------
2. Alpha-Beta Pruning Evaluation:
   - Root Alpha-Beta Value  : 5
   - Total Leaf Evaluations : 5
   - Branches Pruned Count  : 2
     Pruning Event 1: At D (MAX) | Pruned: ['L4'] | Trigger: beta (5) <= alpha (6)
     Pruning Event 2: At B (MIN) | Pruned: ['F'] | Trigger: beta (2) <= alpha (5)
-----------------------------------------------------------------
Comparative Performance Summary:
Algorithm            | Root Score   | Leaves Evaluated   | Leaves Pruned
-----------------------------------------------------------------
Minimax              | 5            | 8                  | 0
Alpha-Beta Pruning   | 5            | 5                  | 3
-----------------------------------------------------------------
Verification Checks:
 [PASS] Root minimax scores match: True
 [PASS